In [9]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md

In [10]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-nano-2025-04-14"
)

In [11]:
# research를 plan 함
research_planner = AssistantAgent(
    "research_planner",
    description="복합적인 질문을 연구 하위 작업으로 세분화하는 전략적 연구 코디네이터",
    model_client=model_client,
    system_message="""당신은 연구 기획 전문가입니다. 당신의 임무는 집중적인 연구 계획을 세우는 것입니다.
    
    각 연구 질문에 대해 다음과 같은 내용을 포함하여 집중적인 연구 계획을 작성하십시오,
    
    1. **핵심 주제:**: 조사해야 할 2~3가지 주요 영역
    2. **검색 쿼리**: 다음 내용을 포함하는 3~5개의 구체적인 검색어 생성:,
       - 최신 동향 및 뉴스,
       - 주요 통계 또는 데이터,
       - 전문가 분석 또는 연구 자료,
       - 향후 전망,
    
    계획은 집중도 있고 달성 가능하게 유지하십시오. 양보다 질이 중요합니다.""",
)

# 실제 tool을 사용해서  이 angent의 plan대로 research 함
research_agent = AssistantAgent(
    "research_agent",
    description="웹 콘텐츠를 검색하고 추출하는 웹 리서치 전문가",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""당신은 웹 리서치 전문가입니다. 당신의 임무는 연구 계획에 따라 집중적인 검색을 수행하는 것입니다.

연구 전략:
1. 연구 계획에 명시된 **검색을 3~5회** 실행합니다.
2. 결과에서 **핵심 정보를 추출**합니다:
    - 주요 사실 및 통계
    - 최근 동향
    - 전문가 의견
    - 중요한 배경 맥락

3. **품질 중심:**:
    - 권위 있는 출처를 우선시합니다.
    - 최근 정보(2년 이내)를 찾습니다.
    - 다양한 관점을 기록합니다.

계획된 검색을 모두 마친 후, 찾아낸 내용을 요약하십시오. 당신의 목표는 5~10개의 양질의 출처를 확보하는 것입니다.""",
)

# research를 분석
research_analyst = AssistantAgent(
    "research_analyst",
    description="연구 보고서를 작성하는 전문 분석가",
    model_client=model_client,
    system_message="""당신은 연구 분석가입니다. 수집된 조사 자료를 바탕으로 포괄적인 보고서를 작성하십시오.

다음 내용을 포함하여 연구 보고서를 작성하십시오:

## 요약 (Executive Summary)
- 주요 조사 결과 및 결론
- 핵심 통찰(인사이트)

## 배경 및 현황 (Background & Current State)
- 현재 상황 및 환경
- 최근 동향
- 주요 통계 및 데이터

## 분석 및 통찰 (Analysis & Insights)
- 주요 트렌드
- 다양한 관점
- 전문가 의견

## 향후 전망 (Future Outlook)
- 신규 트렌드
- 예측
- 시사점

## 출처 (Sources)
- 사용된 모든 출처 목록

Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.""",
)

# 리뷰어
quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="연구의 완결성과 정확성을 평가하는 품질 보증 전문가",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""당신은 품질 검토자입니다. 당신의 임무는 연구 분석가가 완벽한 연구 보고서를 작성했는지 확인하는 것입니다.

다음을 확인하십시오:
- 연구 분석가가 작성한, "REPORT_COMPLETE"로 끝나는 포괄적인 연구 보고서
- 연구 질문에 대해 충분히 답변되었는지 여부
- 출처가 인용되었으며 신뢰할 수 있는지 여부
- T보고서에 요약, 핵심 정보, 분석 및 출처가 포함되어 있는지 여부

"REPORT_COMPLETE"로 끝나는 완성된 연구 보고서를 확인하면 다음과 같이 수행하십시오:
1. 먼저, save_report_to_md 도구를 사용하여 보고서를 report.md에 저장합니다.
2. 그다음 다음과 같이 말하십시오: "조사가 완료되었습니다. 보고서가 report.md에 저장되었습니다. 보고서를 검토해 보시고 승인하실지, 아니면 추가 조사가 필요한지 알려주세요."

연구 분석가가 아직 완성된 보고서를 작성하지 않았다면, 지금 바로 작성하라고 지시하십시오.""",
)

# 재검색 강화 전문가
research_enhancer = AssistantAgent(
    "research_enhancer",
    description="중요한 정보의 공백(누락된 부분)만 찾아내는 전문가",
    model_client=model_client,
    system_message="""당신은 연구 보완 전문가입니다. 당신의 업무는 오직 결정적인 공백만 찾아내는 것입니다.

연구 내용을 검토하고, 다음과 같은 중대한 공백이 있을 때만 추가 검색을 제안하십시오:
- 최근 6개월 이내의 최신 동향이 완전히 누락됨
- 통계나 데이터가 전혀 없음
- 특별히 요청받은 핵심적인 관점이 빠짐

만약 연구가 기초적인 내용을 충분히 다루고 있다면, 다음과 같이 말하십시오: "보고서를 진행하기에 연구 내용이 충분합니다."

반드시 필요한 경우에만 1~2개의 추가 검색을 제안하십시오. 우리는 완벽한 조사보다는 실질적으로 좋은 보고서를 완성하는 것을 우선시합니다.""",
)

user_proxy = UserProxyAgent(
    "user_proxy",
    description="추가 조사를 요청하거나 최종 결과를 승인할 수 있는 사람 검토자",
    input_func=input,
)



In [12]:
selector_prompt = """
대화 기록을 바탕으로 현재 작업에 가장 적합한 에이전트를 선택하십시오: 

{roles}

현재 대화 내용: 
{history}

이용 가능한 에이전트:
- research_planner: 연구 접근 방식 계획 (시작 단계에서만 사용)
- research_agent: 웹 소스 검색 및 콘텐츠 추출 (계획 수립 후 사용)
- research_enhancer: 오직 결정적인 공백만 파악 (제한적으로 사용)
- research_analyst: 최종 연구 보고서 작성
- quality_reviewer: 완성된 보고서 존재 여부 확인
- user_proxy: 사용자에게 피드백 요청

워크플로우:
1. 아직 계획이 수립되지 않은 경우 → research_planner 선택
2. 계획은 수립되었으나 조사가 진행되지 않은 경우 → research_agent 선택
3. research_agent가 초기 검색을 완료한 후 → research_enhancer를 한 번 선택
4. enhancer가 "진행하기에 충분함(sufficient to proceed)"이라고 하면 → research_analyst 선택
5. enhancer가 결정적인 추가 검색을 제안하면 → research_agent를 한 번 더 거친 후 research_analyst 선택
6. research_analyst가 "REPORT_COMPLETE"라고 하면 → quality_reviewer 선택
7. quality_reviewer가 사용자 피드백을 요청하면 → user_proxy 선택

중요: research_agent가 최대 2회 검색을 수행한 후에는 결과와 상관없이 research_analyst로 진행하십시오.

이 워크플로우를 기반으로 다음에 작업할 에이전트를 선택하십시오.
"""

In [13]:


text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst, 
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    #allow_repeated_speaker=True,
    termination_condition=termination_condition,
)

In [ ]:
await Console(team.run_stream(task="원자력 에너지의 새로운 발전에 관한 연구"))

---------- TextMessage (user) ----------
원자력 에너지의 새로운 발전에 관한 연구
---------- TextMessage (research_planner) ----------
**연구 계획: 원자력 에너지의 새로운 발전에 관한 연구**

---

### 1. 핵심 주제
- **신기술 및 혁신적 원자력 발전 방식**: 소형모듈원자로(SMR), 열원·전력계통 연결 신규 기술, 차세대 핵분열/핵융합 기술
- **안전성 및 방사성 폐기물 관리 발전**: 최신 안전 기술, 폐기물 처리 및 재활용 혁신 방안
- **경제성 및 정책 전망**: 원자력 발전의 비용 경쟁력, 규제 변화, 글로벌 정책 동향

---

### 2. 검색 쿼리

#### 최신 동향 및 뉴스
- `"latest innovations in nuclear energy 2024" OR "recent developments in small modular reactors" OR "advances in nuclear safety technology"`
- `"global nuclear energy news 2023" OR "nuclear power breakthroughs news"`
- `"government policies on nuclear energy 2024" OR "new nuclear regulation updates"`

#### 주요 통계 및 데이터
- `"nuclear energy capacity statistics 2023" OR "global nuclear power plants data" OR "nuclear energy market size forecast 2025"`
- `"cost analysis of new nuclear reactor technologies" OR "nuclear waste volume statistics"`
- `"investment trends in nuclear energy 2023" OR "public acceptance 